In [ ]:
# hide
import numpy as np
import matplotlib.pyplot as plt
import pyquist as pq
from scipy.signal import lfilter

In [ ]:
# hide
def lpc(x: np.ndarray, K: int) -> np.ndarray:
    """The coefficients h_1, ..., h_K that best predict each x[n] from x[n-1], ..., x[n-K]."""
    N = len(x)
    X = np.stack([x[K - k:N - k] for k in range(1, K + 1)], axis=1)   # row n: x[n-1], ..., x[n-K]
    h, *_ = np.linalg.lstsq(X, x[K:], rcond=None)                     # minimize the squared error
    return h

In [ ]:
vowel = pq.Audio.from_file("../assets/vowel-a.wav")
f_s = vowel.sample_rate
x = vowel.samples[:, 0].astype(np.float64)
h = lpc(x[f_s:f_s + f_s // 2], K=40)

# SciPy's lfilter(b, a, x) computes y[n] = sum_k b[k] x[n-k] - sum_{k>0} a[k] y[n-k]
coefficients = np.concatenate([[1.0], -h])     # [1, -h_1, ..., -h_K]
e = lfilter(coefficients, [1.0], x)             # e[n] = x[n] - sum_k h_k x[n-k]
y = lfilter([1.0], coefficients, e)             # y[n] = e[n] + sum_k h_k y[n-k]
print(f"Largest difference between x and y: {np.max(np.abs(x - y)):.1e}")

t = 1000 * np.arange(f_s, f_s + 2000) / f_s
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].plot(t, x[f_s:f_s + 2000])
axes[0].set_ylabel("$x[n]$")
axes[1].plot(t, e[f_s:f_s + 2000], color="C3")
axes[1].set_ylabel("$e[n]$")
axes[1].set_xlabel("Time (ms)")
plt.show()
pq.play(pq.Audio(e, f_s), normalize=True)